# **New York Runner**

## **1. Project Setup**

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\candle-fetcher


## **2. Imports**

In [2]:
from src.pipeline import run_new_york_pipeline, print_pipeline_result
from src.mt5_fetcher import MT5FetchConfig, fetch_raw_candles, print_fetch_result

## **3. Settings**

In [3]:
# ---------------------------------------------------------------------------
# New York Runner Settings
# ---------------------------------------------------------------------------

SYMBOL = "US100.cash"
TIMEFRAME = "1m"

START_DATE = "2026-07-27"
END_DATE = "2026-08-21"

# Optional filters.
# Leave as None if you do not want to use them.
YEARS = None
MONTHS = None
WEEKS = None
DAYS = None
WEEKDAYS = None

OUTPUT_SUFFIX = None
OUTPUT_NAME = None


# ---------------------------------------------------------------------------
# MT5 Fetch Settings
# ---------------------------------------------------------------------------

FETCH_FROM_MT5 = True

RANGE_NAME = "new_york_range"
RAW_OUTPUT_NAME = None

FETCH_START_DATE = START_DATE
FETCH_END_DATE = END_DATE

CHUNK_DAYS = 31

# Dates are interpreted in this timezone before converting to UTC.
DATE_TIMEZONE_NAME = "America/New_York"

## **4. Run New York Filter**

In [4]:
input_pattern = "*.csv"

if FETCH_FROM_MT5:
    fetch_config = MT5FetchConfig(
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        range_name=RANGE_NAME,
        output_name=RAW_OUTPUT_NAME,
        start_date=FETCH_START_DATE,
        end_date=FETCH_END_DATE,
        chunk_days=CHUNK_DAYS,
        date_timezone_name=DATE_TIMEZONE_NAME,
    )

    fetch_result = fetch_raw_candles(fetch_config)
    print_fetch_result(fetch_result)

    # Only process the CSV that was just fetched.
    input_pattern = fetch_result["filename"]

else:
    print("Skipping MT5 fetch. Loading existing CSV files from data/raw/.")

Fetching chunk: 2026-07-27 04:00 -> 2026-08-22 04:00 UTC
MT5 raw candle fetch complete
Symbol: US100.cash
Timeframe: M1
Rows saved: 27,125
First datetime UTC: 2026-07-27 04:00:00+00:00
Last datetime UTC: 2026-08-21 23:49:00+00:00
Filename: us100_cash_M1_raw_new_york_range_2026-07-27_to_2026-08-21.csv
Path: C:\candle-fetcher\data\raw\us100_cash_M1_raw_new_york_range_2026-07-27_to_2026-08-21.csv


## **5. Run New York Filter**

In [5]:
result = run_new_york_pipeline(
    start_date=START_DATE,
    end_date=END_DATE,
    years=YEARS,
    months=MONTHS,
    weeks=WEEKS,
    days=DAYS,
    weekdays=WEEKDAYS,
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    input_pattern=input_pattern,
    output_suffix=OUTPUT_SUFFIX,
    output_name=OUTPUT_NAME,
)

print_pipeline_result(result)

Starting candle pipeline
New York: 09:10 to 12:30 (America/New_York)
Loading: C:\candle-fetcher\data\raw\us100_cash_M1_raw_new_york_range_2026-07-27_to_2026-08-21.csv
Raw rows loaded: 27,125
Raw first datetime: 2026-07-27 04:00:00+00:00
Raw last datetime: 2026-08-21 23:49:00+00:00
Rows after date filters: 27,125
Rows after session filter: 4,000
Export complete
Session: new_york
Rows saved: 4,000
Filename: us100_cash_M1_new_york_2026_07_27_to_2026_08_21.csv
Path: C:\candle-fetcher\data\processed\new_york\us100_cash_M1_new_york_2026_07_27_to_2026_08_21.csv
Pipeline result
Session: new_york
Description: New York: 09:10 to 12:30 (America/New_York)
Raw rows: 27,125
Date-filtered rows: 27,125
Final rows: 4,000
First datetime: 2026-07-27 13:10:00+00:00
Last datetime: 2026-08-21 16:29:00+00:00
Saved to: C:\candle-fetcher\data\processed\new_york\us100_cash_M1_new_york_2026_07_27_to_2026_08_21.csv


## **6. View Data**

In [6]:
df = result["data"]
df.head()

,datetime,open,high,low,close,tick_volume,spread,real_volume,session_name
0,2026-07-27 13:10:00+00:00,28583.63,28584.53,28572.13,28573.33,288,145,0,new_york
1,2026-07-27 13:11:00+00:00,28573.23,28580.13,28568.13,28568.33,293,155,0,new_york
2,2026-07-27 13:12:00+00:00,28568.63,28571.93,28567.33,28570.03,234,165,0,new_york
3,2026-07-27 13:13:00+00:00,28570.13,28576.93,28569.43,28573.43,224,165,0,new_york
4,2026-07-27 13:14:00+00:00,28573.63,28582.33,28572.13,28578.93,202,165,0,new_york


## **7. Check Size**

In [7]:
df.shape

(4000, 9)